# API Schema RAG — Phase 2: Request/Response Detail
Phase 1 answered *'which endpoint does X?'* Phase 2 answers *'how do I call it?'* — the **parameters, request body, response fields, and error codes** for each endpoint.

The developer's questions here are aspect-scoped: *'what fields does the refund body take?'* wants only the request-body chunk; *'what errors can create-payment return?'* wants only the errors chunk. This notebook builds retrieval that answers exactly those.

## The key design difference from Phase 1
| | Phase 1 (discovery) | Phase 2 (schema) |
|---|---|---|
| Rows per endpoint | **one** (its purpose) | **several** (one per aspect) |
| What's embedded | the purpose | each schema aspect separately |
| New metadata | — | **`aspect`** = params / request / response / errors |
| Query shape | 'which API does X' | 'what are the {aspect} of endpoint Y' |

**Why split one endpoint into several chunks?** If you embedded an endpoint's entire schema as one blob, every query — 'what errors?', 'what fields?' — would retrieve the same giant chunk and force the LLM to hunt inside it. Splitting by aspect makes retrieval precise and lets you **filter by aspect** in Chroma Studio.

## Setup
`USE_MOCK=True` runs offline. Set `False` for real in-house Jina embeddings (schema retrieval quality is much better with the real model).

In [ ]:
USE_MOCK = True   # set False for real InHouseEmbeddings()

import chromadb, pandas as pd, numpy as np, re

if USE_MOCK:
    _STOP = set("the a an to of and or is are be for in on at by with from your you we our "
                "it its as within once after before per must can may that this no all".split())
    class MockEmbedder:
        def _vec(self, t):
            v = np.zeros(256)
            for w in re.findall(r"[a-z0-9]+", t.lower()):
                if w not in _STOP and len(w) > 2:
                    v[abs(hash(w)) % 256] += 1
            n = np.linalg.norm(v); return (v/n if n else v)
        def __call__(self, input):
            if isinstance(input, str): input = [input]
            return [self._vec(t).tolist() for t in input]
        def embed_documents(self, texts): return self(texts)
        def embed_query(self, text): return self._vec(text).tolist()
    embedder = MockEmbedder()
    print("MOCK embedder. Enough to see aspect-scoped retrieval + filtering behavior.")
else:
    from inhouse_wrappers import InHouseEmbeddings
    embedder = InHouseEmbeddings()
    print("Real InHouseEmbeddings().")

## 1. Load the schema chunks

In [ ]:
df = pd.read_csv("dataset/api_schema_chunks.csv")
print(f"{len(df)} schema chunks across {df['endpoint'].nunique()} endpoints")
print("Aspects:", dict(df["aspect"].value_counts()))
META_FIELDS = ["endpoint", "method", "domain", "aspect", "version", "status"]
df.head(8)

## 2. Ingest — each aspect chunk is its own document
Note there are multiple rows per endpoint now, distinguished by the `aspect` field.

In [ ]:
client = chromadb.PersistentClient(path="./api_schema_db")
try: client.delete_collection("api_schema")
except Exception: pass
coll = client.get_or_create_collection("api_schema", metadata={"hnsw:space": "cosine"})

ids = df["id"].tolist()
docs = df["text"].tolist()
metadatas = [{f: row[f] for f in META_FIELDS} for _, row in df.iterrows()]
coll.add(ids=ids, embeddings=embedder.embed_documents(docs), documents=docs, metadatas=metadatas)
print(f"Ingested {coll.count()} chunks.")

# show all aspects for ONE endpoint to make the chunking concrete
r = coll.get(where={"endpoint": "/v1/payments/{id}/refund"}, include=["documents","metadatas"])
print(f"\nThe refund endpoint expands into {len(r['ids'])} chunks:")
for i, m in zip(r["ids"], r["metadatas"]):
    print(f"   {m['aspect']:9s} -> {i}")

## 3. Aspect-scoped retrieval — the core Phase 2 capability
The developer names an aspect ('fields', 'errors', 'parameters') and an endpoint or capability. We detect the aspect and filter to it, so retrieval returns only the relevant schema chunk.

In [ ]:
ASPECT_HINTS = {
    "errors":   ["error", "fail", "declined", "exception", "status code", "40"],
    "params":   ["parameter", "path param", "query param", "url path"],
    "request":  ["body", "field", "input", "payload", "send in the request", "request body"],
    "response": ["response", "returns", "return", "output", "get back", "retrieve", "success response", "what does it return"],
}
def detect_aspect(query):
    # order matters: check the more-specific aspects (errors, params) before the
    # broad ones (request/response), since words like "return" appear in several.
    q = query.lower()
    for aspect, hints in ASPECT_HINTS.items():
        if any(h in q for h in hints):
            return aspect
    return None

def schema_lookup(query, k=3, endpoint=None):
    where_terms = []
    aspect = detect_aspect(query)
    if aspect:  where_terms.append({"aspect": aspect})
    if endpoint: where_terms.append({"endpoint": endpoint})
    where = None
    if len(where_terms) == 1: where = where_terms[0]
    elif len(where_terms) > 1: where = {"$and": where_terms}

    kw = {"query_embeddings": [embedder.embed_query(query)], "n_results": k,
          "include": ["documents","metadatas","distances"]}
    if where: kw["where"] = where
    res = coll.query(**kw)
    rows = []
    for i in range(len(res["ids"][0])):
        rows.append({"id": res["ids"][0][i],
                     "aspect": res["metadatas"][0][i]["aspect"],
                     "endpoint": res["metadatas"][0][i]["endpoint"],
                     "text": res["documents"][0][i][:90] + "...",
                     "dist": round(res["distances"][0][i], 3)})
    return aspect, pd.DataFrame(rows)

aspect, out = schema_lookup("what fields does the refund request body take?")
print("Detected aspect:", aspect)
out

**What happened:** the query said 'fields'/'body', so we detected `aspect=request` and filtered to only request-body chunks — the refund request chunk should top the list, not its errors or response chunk. That precision is the whole point of chunking by aspect.

### Try several aspect-scoped questions

In [ ]:
for q in [
    "what errors can create payment return?",
    "what does the get payment endpoint return?",
    "what parameters does freeze card take?",
    "what fields are in the issue card request body?",
]:
    aspect, out = schema_lookup(q, k=1)
    top = out.iloc[0]
    print(f"[{aspect or 'any':8s}] {q}")
    print(f"          -> {top['id']} ({top['endpoint']})")

## 4. Pin to a specific endpoint
When the developer already knows the endpoint, pass it explicitly to combine endpoint + aspect filters — the most precise lookup.

In [ ]:
aspect, out = schema_lookup("what body fields?", endpoint="/v1/payments", k=3)
print(f"Endpoint=/v1/payments, aspect={aspect}:")
display(out)

## 5. Reassemble the full schema for one endpoint
Sometimes you want the *whole* schema card, not one aspect — e.g. to render docs. Just get all chunks for an endpoint, ordered by aspect.

In [ ]:
def full_schema(endpoint):
    r = coll.get(where={"endpoint": endpoint}, include=["documents","metadatas"])
    order = {"params":0, "request":1, "response":2, "errors":3}
    items = sorted(zip(r["metadatas"], r["documents"]), key=lambda x: order.get(x[0]["aspect"], 9))
    print(f"=== {endpoint} ===")
    for m, doc in items:
        print(f"\n[{m['aspect'].upper()}]")
        print(" ", doc)

full_schema("/v1/payments/{id}/refund")

## 6. Examination in Chroma Studio
Point Studio at this collection and the `aspect` field becomes a first-class filter.

In [ ]:
def scan_metadata(coll):
    got = coll.get(limit=coll.count(), include=["metadatas"]); metas = got["metadatas"]
    keys = sorted({k for m in metas if m for k in m.keys()}); summary={}
    for k in keys:
        c={}
        for m in metas:
            if m and k in m: c[str(m[k])]=c.get(str(m[k]),0)+1
        summary[k]=dict(sorted(c.items(), key=lambda x:-x[1]))
    return summary
for k,v in scan_metadata(coll).items(): print(f"{k:9s}: {v}")

import os
print("\nOpen in Chroma Studio -> Local Chroma folder:")
print("  ", os.path.abspath("./api_schema_db"))
print("  select collection 'api_schema'")

**In Chroma Studio, try:**
- **Browse → filter `aspect = errors`** → see every error chunk across all endpoints in one view (great for an 'all error codes' audit).
- **Visualize → color by `aspect`** → params/request/response/errors should form four visible families; chunks of the same aspect across endpoints cluster by their shared vocabulary.
- **Visualize → color by `domain`, filter `aspect = request`** → compare request-body shapes across domains.
- **Browse → filter `endpoint = /v1/payments/{id}/refund`** → the 4 chunks that make up that endpoint's schema.

## 7. How Phase 1 and Phase 2 work together
In a real assistant you'd chain them:
1. **Phase 1** discovers the endpoint from a capability query ('refund a payment' → `/v1/payments/{id}/refund`).
2. **Phase 2** answers follow-up schema questions about that endpoint ('what fields does its body take?') by filtering `endpoint` + `aspect`.

They're deliberately separate collections because the chunking is different (one purpose-doc per endpoint vs several schema-chunks per endpoint) — mixing them in one collection would blur discovery and schema retrieval. The shared `endpoint` metadata is the join key between them.

In [ ]:
# Illustrative chain (Phase 1 result feeds Phase 2). Here we just hardcode the
# Phase-1 endpoint; in the full system Phase 1's discover() would return it.
discovered_endpoint = "/v1/payments/{id}/refund"     # <- would come from Phase 1
print("Phase 1 discovered:", discovered_endpoint)
print("\nPhase 2 follow-up: 'what body fields does it take?'")
aspect, out = schema_lookup("what body fields does it take?", endpoint=discovered_endpoint, k=2)
display(out)